# 02 - Augmentation (مطابق لـ `augmentation.py`)

هنا بنذاكر الجزء اللي بياخد نتائج retrieval ويحوّلها إلى Evidence Packet يفهمه الـ LLM.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'rag' else Path.cwd()))
import inspect
from rag.retrieval import retrieve
from rag import augmentation

print('Production module:', augmentation.__file__)
print('Max context default:', inspect.signature(augmentation.build_context))

## 1. كود بناء الـ context
`build_context()` يضيف tags مثل `[S1]`، يحافظ على source metadata، ويوقف النص عند `max_chars=9000`. 

In [ ]:
print(inspect.getsource(augmentation.build_context))

## 2. تطبيق الكود على retrieval حقيقي

In [ ]:
question = 'Explain S34 in Arabic'
records = retrieve('ankh and life', codes=['S34'], top_k=4)
packet = augmentation.build_context(question, records)
print('--- Evidence context ---')
print(packet['context'])
print('\n--- Citations ---')
for citation in packet['citations']:
    print(citation)
print('\n--- Instructions ---')
print(packet['instructions'])
print('\nConfidence:', packet['confidence'])

## 3. شكل رسائل الـ provider
الدالة دي لا تستدعي Groq؛ هي فقط تجهز system/user messages، و`chatbot.py` هو اللي يرسلها للـ provider.

In [ ]:
print(inspect.getsource(augmentation.prompt_messages))
messages = augmentation.prompt_messages(question, records)
print('Roles:', [message['role'] for message in messages])
print('User packet keys:', list(__import__('json').loads(messages[1]['content']).keys()))